# Clasificador Sanrio 12 Clases - ResNet18 Transfer Learning v5 (Optimizacion)

**Diplomado Superior RNA y Deep Learning - UAEM Ecatepec**  
**Modulo 4: Deep Learning | Proyecto Final**  
**Autora:** Diana Blanco - MorritaConP1to

---

### Historial de versiones

| Version | Arquitectura | Dataset | Clases | Accuracy | Problema |
|---------|-------------|--------|:------:|:--------:|---------|
| v1 | CNN desde cero | 10 originales | 10 | ~82% | Modelo muy simple |
| v2 | ResNet18 + 2 capas | 29 clases (ruidoso) | 29 | **65.49%** | Muchas clases similares, dataset contaminado |
| v3 | EfficientNet-B0 + 1 capa | v3 (12 curadas) | 12 | **66.43%** | Cabeza muy simple, batch 32, sin limpiar |
| **v4** | **ResNet18 + 2 capas** | **v4 (12 limpias)** | **12** | **88.54%** | **Mejor modelo (baseline)** |
| v5.1 (MixUp) | ResNet18 + MixUp a=0.2 | v4 | 12 | 87.50% | MixUp anadio ruido, empeoro |
| **v5.2 (FocalLoss)** | **ResNet18 + FocalLoss g=2.0** | **v4** | **12** | **88.24%** | **No mejoro vs v4** |

### Cambios vs v4

| Aspecto | v4 (baseline) | v5 (optimizacion) | Resultado |
|---------|:-------------:|:-----------------:|:--------:|
| Loss | CrossEntropy + label_smoothing | **FocalLoss g=2.0 + label_smoothing** | Enfoca en hard examples |
| Dropout cabeza | 0.3 | **0.5** | Mas regularizacion |
| Fase 3 epocas | 20 | **30** | Mas fine-tuning |
| MixUp | Ausente | **Ausente** | Se descarto (empeora) |

### Conclusion de optimizacion

Ninguna optimizacion supero a v4 (88.54%). FocalLoss con dropout 0.5 y 30 epocas
dio 88.24%. La configuracion v4 es la optima para este dataset.

### Las 12 clases (dataset v4)

| Personaje | Train | Test |
|-----------|------:|-----:|
| aggretsuko | 199 | 50 |
| badtz_maru | 199 | 51 |
| cinnamon | 272 | 69 |
| gudetama | 260 | 65 |
| hangyodon | 237 | 59 |
| hello_kitty | 229 | 57 |
| keroppi | 234 | 60 |
| kirimichan | 144 | 37 |
| kuromi | 249 | 63 |
| my_melody | 188 | 46 |
| pochacco | 234 | 57 |
| pompompurin | 228 | 58 |
| **Total** | **2,673** | **672** |

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELDA 1 — CONFIG + SETUP                                    ║
# ║                                                              ║
# ║  Para cambiar de version del dataset, solo edita:            ║
# ║    DATASET_VERSION = 'v4'  → train_v4/ + test_v4/            ║
# ║                                                              ║
# ║  Las clases se detectan automaticamente de las carpetas.     ║
# ╚══════════════════════════════════════════════════════════════╝

import os, sys, time, json, warnings, gc, copy, math
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from datetime import datetime

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler, TensorDataset
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
import torchvision.models as models
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

warnings.filterwarnings('ignore')

AUTORA  = 'Diana Blanco — MorritaConP1to'
VERSION = 'v5-ResNet18-12clases-FocalLoss'
SEMILLA = 42
INICIO  = time.time()
torch.manual_seed(SEMILLA)
np.random.seed(SEMILLA)

# ── Version del dataset ──
DATASET_VERSION = 'v4'

# ── Deteccion de rutas ──
cwd = os.getcwd()
if os.path.basename(cwd) == 'notebooks':
    RUTA_BASE = os.path.dirname(cwd)
elif os.path.isdir(os.path.join(cwd, 'dataset')):
    RUTA_BASE = cwd
else:
    RUTA_BASE = cwd

RUTA_DATASET = os.path.join(RUTA_BASE, 'dataset')
RUTA_MODELOS = os.path.join(RUTA_BASE, 'modelos')
RUTA_MODELS  = os.path.join(RUTA_BASE, 'models')
TRAIN_DIR    = os.path.join(RUTA_DATASET, f'train_{DATASET_VERSION}')
TEST_DIR     = os.path.join(RUTA_DATASET, f'test_{DATASET_VERSION}')
os.makedirs(RUTA_MODELOS, exist_ok=True)
os.makedirs(RUTA_MODELS,  exist_ok=True)

# ── Hiperparametros ──
# v4 usa batch=16 (mejor que 32 para datasets chicos)
# y 3 fases para transicion gradual
CONFIG = {
    'BATCH_SIZE':      16,
    'EPOCHS_FASE1':    10,
    'EPOCHS_FASE2':    15,
    'EPOCHS_FASE3':    30,
    'LR_FASE1':        0.01,
    'LR_FASE2':        0.001,
    'LR_FASE3':        0.0005,
    'DROPOUT':         0.5,
    'PATIENCE':        5,
    'MOMENTUM':        0.9,
    'WEIGHT_DECAY':    1e-4,
    'GRAD_CLIP':       1.0,
    'LABEL_SMOOTHING': 0.1,
    'FOCAL_GAMMA':     2.0,
}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if not os.path.isdir(TRAIN_DIR):
    raise FileNotFoundError(
        f'No se encuentra: {TRAIN_DIR}\n'
        f'Ejecuta scraping/limpiar_dataset_v4.py primero.'
    )

print(f'Device:          {device}')
if device.type == 'cuda':
    print(f'GPU:             {torch.cuda.get_device_name(0)}')
print(f'Dataset version: {DATASET_VERSION}')
print(f'Train dir:       {TRAIN_DIR}')
print(f'Test dir:        {TEST_DIR}')
print(f'CONFIG:          {json.dumps(CONFIG, indent=14)}')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELDA 2 — DATASET + AUGMENTACION + LOADERS                  ║
# ╚══════════════════════════════════════════════════════════════╝

transform_train = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.RandomGrayscale(p=0.10),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.10, scale=(0.02, 0.15)),
])

transform_test = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_dataset = ImageFolder(root=TRAIN_DIR, transform=transform_train)
test_dataset  = ImageFolder(root=TEST_DIR,  transform=transform_test)
clases        = train_dataset.classes
NUM_CLASES    = len(clases)

print(f'Clases detectadas ({NUM_CLASES}): {clases}')

# Guardar JSON de clases para el backend
ruta_clases = os.path.join(RUTA_MODELOS, f'clases_sanrio_{DATASET_VERSION}.json')
with open(ruta_clases, 'w', encoding='utf-8') as f:
    json.dump(clases, f, ensure_ascii=False, indent=2)
print(f'Clases guardadas: {ruta_clases}')

# Pesos por clase
conteos = [
    len([f for f in os.listdir(os.path.join(TRAIN_DIR, c))
         if os.path.isfile(os.path.join(TRAIN_DIR, c, f))])
    for c in clases
]
total_imgs  = sum(conteos)
pesos_clase = torch.tensor([total_imgs / max(n, 1) for n in conteos],
                           dtype=torch.float).to(device)

print('\nDistribucion train:')
for c, n in sorted(zip(clases, conteos), key=lambda x: x[1]):
    barra = '#' * max(1, n // 8)
    warning = '  MENOS IMAGENES' if n < 200 else ''
    print(f'  {c:20s}: {n:4d} img  {barra}{warning}')
print(f'\n  Total: {total_imgs}')

sampler = WeightedRandomSampler(
    weights=[pesos_clase[train_dataset.targets[i]].item() for i in range(len(train_dataset))],
    num_samples=len(train_dataset), replacement=True
)

train_loader = DataLoader(train_dataset, batch_size=CONFIG['BATCH_SIZE'],
                          sampler=sampler, num_workers=0,
                          pin_memory=(device.type=='cuda'))
test_loader  = DataLoader(test_dataset,  batch_size=CONFIG['BATCH_SIZE'],
                          shuffle=False, num_workers=0,
                          pin_memory=(device.type=='cuda'))
print(f'Train batches: {len(train_loader)} | Test batches: {len(test_loader)}')

## Arquitectura: ResNet18 + Cabeza de 2 capas

**Por que ResNet18 y no EfficientNet-B0?**

| Aspecto | ResNet18 | EfficientNet-B0 |
|---------|:--------:|:---------------:|
| Parametros totales | 11.2M | 5.3M |
| Parametros cabeza | ~132K (2 capas) | ~33K (1 capa) |
| Accuracy v2 | 65.49% (29 clases) | 66.43% (26 clases) |
| Accuracy por clase | Mejor distribucion | Muy desigual (my_melody 31%) |

La cabeza de **2 capas con ReLU intermedio** permite aprender combinaciones
no-lineales de los features. La cabeza de 1 capa de EfficientNet asume que
los features finales son linealmente separables, lo cual es falso para
personajes Sanrio visualmente similares.

**Estrategia de 3 fases:**
- Fase 1: Solo cabeza (10 epocas, lr=0.01)
- Fase 2: Capa 4 + cabeza (15 epocas, lr=0.001)
- Fase 3: Capas 3 y 4 + cabeza (20 epocas, lr=0.0005)

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELDA 3 — MODELO ResNet18 + CABEZA DE 2 CAPAS              ║
# ╚══════════════════════════════════════════════════════════════╝

weights = models.ResNet18_Weights.IMAGENET1K_V1
model   = models.resnet18(weights=weights)

# Congelar todo el backbone
for param in model.parameters():
    param.requires_grad = False

# Cabeza de 2 capas (como en v2 que funciono mejor)
# 512 → 256 → NUM_CLASES con ReLU + Dropout intermedio
in_features = model.fc.in_features  # 512 para ResNet18
model.fc = nn.Sequential(
    nn.Dropout(p=CONFIG['DROPOUT']),
    nn.Linear(in_features, 256),
    nn.ReLU(inplace=True),
    nn.Dropout(p=0.2),
    nn.Linear(256, NUM_CLASES),
)
model = model.to(device)

# Loss con label smoothing (aprendido de v3)
# Focal Loss (v5)
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None, label_smoothing=0.0):
        super().__init__()
        self.gamma = gamma
        self.weight = weight
        self.label_smoothing = label_smoothing
    def forward(self, input, target):
        ce_loss = F.cross_entropy(input, target, weight=self.weight,
                                  label_smoothing=self.label_smoothing,
                                  reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma * ce_loss).mean()
        return focal_loss

criterion = FocalLoss(gamma=CONFIG['FOCAL_GAMMA'],
              weight=pesos_clase,
              label_smoothing=CONFIG['LABEL_SMOOTHING'])
print(f'Loss function: FocalLoss(gamma={CONFIG['FOCAL_GAMMA']})')


total_p   = sum(p.numel() for p in model.parameters())
entrena_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Parametros totales:     {total_p:,}')
print(f'Parametros entrenables: {entrena_p:,} ({100*entrena_p/total_p:.1f}%)')
print(f'Clases de salida:       {NUM_CLASES}')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELDA 4 — FUNCION DE ENTRENAMIENTO REUTILIZABLE             ║
# ╚══════════════════════════════════════════════════════════════╝

def entrenar(modelo, train_loader, test_loader, criterion,
             optimizer, scheduler, num_epochs, device, fase):
    historial  = {'train_loss': [], 'test_loss': [], 'test_acc': []}
    mejor_loss = float('inf')
    mejor_state = None
    sin_mejora = 0
    t0 = time.time()

    for epoch in range(num_epochs):
        modelo.train()
        tl = 0.0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(modelo(imgs), labels)
            loss.backward()
            nn.utils.clip_grad_norm_(modelo.parameters(), CONFIG['GRAD_CLIP'])
            optimizer.step()
            tl += loss.item()
        tl /= len(train_loader)

        modelo.eval()
        vl = 0.0
        ok = 0
        tot = 0
        with torch.no_grad():
            for imgs, labels in test_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                out = modelo(imgs)
                vl += criterion(out, labels).item()
                _, p = torch.max(out, 1)
                tot += labels.size(0)
                ok += (p == labels).sum().item()
        vl /= len(test_loader)
        acc = 100.0 * ok / tot

        historial['train_loss'].append(tl)
        historial['test_loss'].append(vl)
        historial['test_acc'].append(acc)

        print(f'{fase} [{epoch+1:2d}/{num_epochs}]  '
              f'T:{tl:.4f}  V:{vl:.4f}  Acc:{acc:.2f}%  '
              f'LR:{optimizer.param_groups[0]["lr"]:.6f}  {time.time()-t0:.0f}s')

        if scheduler:
            scheduler.step()

        if vl < mejor_loss:
            mejor_loss = vl
            mejor_state = copy.deepcopy(modelo.state_dict())
            sin_mejora = 0
        else:
            sin_mejora += 1
            if sin_mejora >= CONFIG['PATIENCE']:
                print(f'  Early stopping en epoca {epoch+1}')
                break

    modelo.load_state_dict(mejor_state)
    print(f'  Mejor estado restaurado (val_loss={mejor_loss:.4f})')
    return historial

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELDA 5 — FASE 1: SOLO CABEZA                               ║
# ║  Todos los bloques congelados, solo entrena el nuevo fc      ║
# ╚══════════════════════════════════════════════════════════════╝

print('='*65)
print('FASE 1 — SOLO CABEZA (backbone congelado)')
print('='*65)

opt1 = optim.SGD(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=CONFIG['LR_FASE1'], momentum=CONFIG['MOMENTUM'],
    weight_decay=CONFIG['WEIGHT_DECAY'], nesterov=True
)
sch1 = optim.lr_scheduler.CosineAnnealingLR(opt1, T_max=CONFIG['EPOCHS_FASE1'])

hist1 = entrenar(model, train_loader, test_loader, criterion,
                 opt1, sch1, CONFIG['EPOCHS_FASE1'], device, 'Fase1')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELDA 6 — FASE 2: CAPA 4 + CABEZA                          ║
# ║  Descongelamos layer4 para ajustar detalles de alto nivel    ║
# ╚══════════════════════════════════════════════════════════════╝

print('='*65)
print('FASE 2 — CAPA 4 + CABEZA (layer4 descongelado)')
print('='*65)

for param in model.layer4.parameters():
    param.requires_grad = True

opt2 = optim.SGD([
    {'params': model.layer4.parameters(), 'lr': CONFIG['LR_FASE2']},
    {'params': model.fc.parameters(), 'lr': CONFIG['LR_FASE2']},
], momentum=CONFIG['MOMENTUM'], weight_decay=CONFIG['WEIGHT_DECAY'], nesterov=True)
sch2 = optim.lr_scheduler.CosineAnnealingLR(opt2, T_max=CONFIG['EPOCHS_FASE2'])

entrena_f2 = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_p = sum(p.numel() for p in model.parameters())
print(f'Parametros entrenables: {entrena_f2:,} ({100*entrena_f2/total_p:.1f}%)\n')

hist2 = entrenar(model, train_loader, test_loader, criterion,
                 opt2, sch2, CONFIG['EPOCHS_FASE2'], device, 'Fase2')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELDA 7 — FASE 3: CAPAS 3 Y 4 + CABEZA                     ║
# ║  Descongelamos layer3 + layer4 con LR reducida               ║
# ╚══════════════════════════════════════════════════════════════╝

print('='*65)
print('FASE 3 — CAPAS 3 Y 4 + CABEZA (layer3+layer4 descongelados)')
print('='*65)

for param in model.layer3.parameters():
    param.requires_grad = True

opt3 = optim.SGD([
    {'params': model.layer3.parameters(), 'lr': CONFIG['LR_FASE3'] * 0.5},
    {'params': model.layer4.parameters(), 'lr': CONFIG['LR_FASE3']},
    {'params': model.fc.parameters(),      'lr': CONFIG['LR_FASE3']},
], momentum=CONFIG['MOMENTUM'], weight_decay=CONFIG['WEIGHT_DECAY'], nesterov=True)
sch3 = optim.lr_scheduler.CosineAnnealingLR(opt3, T_max=CONFIG['EPOCHS_FASE3'])

entrena_f3 = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Parametros entrenables: {entrena_f3:,} ({100*entrena_f3/total_p:.1f}%)\n')

hist3 = entrenar(model, train_loader, test_loader, criterion,
                 opt3, sch3, CONFIG['EPOCHS_FASE3'], device, 'Fase3')

historial = {
    'train_loss': hist1['train_loss'] + hist2['train_loss'] + hist3['train_loss'],
    'test_loss':  hist1['test_loss']  + hist2['test_loss']  + hist3['test_loss'],
    'test_acc':   hist1['test_acc']   + hist2['test_acc']   + hist3['test_acc'],
}
print(f'\nMejor accuracy: {max(historial["test_acc"]):.2f}%')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELDA 8 — GUARDAR MODELO + GRAFICAS                         ║
# ╚══════════════════════════════════════════════════════════════╝

ruta_pth = os.path.join(RUTA_MODELOS, f'tl_sanrio_{DATASET_VERSION}_final.pth')
torch.save(model.state_dict(), ruta_pth)
print(f'Pesos guardados: {ruta_pth}')

mejor_acc = max(historial['test_acc'])

metadata = {
    'version': VERSION,
    'dataset_version': DATASET_VERSION,
    'arquitectura': 'ResNet18',
    'num_clases': NUM_CLASES,
    'clases': clases,
    'mejor_accuracy': mejor_acc,
    'accuracy_final': historial['test_acc'][-1],
    'config': {k: v for k, v in CONFIG.items()},
    'baselines': {
        'v2_29cls': 65.49,
        'v3_26cls': 66.43,
    },
    'limpieza': {
        'duplicados_eliminados': 4,
        'leakage_eliminado': 0,
        'imagenes_menos_200px': 30,
        'train_final': 2673,
        'test_final': 672,
    },
    'fecha': datetime.now().strftime('%Y-%m-%d %H:%M'),
}
with open(os.path.join(RUTA_MODELOS, f'metadata_{DATASET_VERSION}.json'), 'w') as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)
print(f'Metadata guardada')

# Graficas
epocas = range(1, len(historial['train_loss']) + 1)
sep1 = len(hist1['train_loss'])
sep2 = sep1 + len(hist2['train_loss'])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'ResNet18 — {NUM_CLASES} clases Sanrio | Dataset {DATASET_VERSION}',
             fontsize=13, fontweight='bold')

ax1.plot(epocas, historial['train_loss'], color='#7F77DD', linewidth=2, label='Train Loss')
ax1.plot(epocas, historial['test_loss'],  color='#D85A30', linewidth=2, label='Val Loss')
ax1.axvline(x=sep1, color='gray', linestyle=':', alpha=0.4)
ax1.axvline(x=sep2, color='gray', linestyle=':', alpha=0.4)
ax1.set_title('Loss')
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(epocas, historial['test_acc'], color='#1D9E75', linewidth=2, label='Val Acc')
ax2.axvline(x=sep1, color='gray', linestyle=':', alpha=0.4)
ax2.axvline(x=sep2, color='gray', linestyle=':', alpha=0.4)
ax2.axhline(y=66.43, color='#D85A30', linestyle=':', alpha=0.7, label='v3 baseline (66.43%)')
ax2.set_title('Accuracy')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(RUTA_MODELOS, f'historial_{DATASET_VERSION}.png'), dpi=120)
plt.show()

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELDA 9 — EVALUACION DETALLADA                              ║
# ╚══════════════════════════════════════════════════════════════╝

plt.close('all'); gc.collect()
if device.type == 'cuda':
    torch.cuda.empty_cache()

model.eval()
y_real, y_pred = [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        _, pred = torch.max(model(imgs.to(device)), 1)
        y_real.extend(labels.cpu().numpy())
        y_pred.extend(pred.cpu().numpy())

# Matriz de confusion
cm = confusion_matrix(y_real, y_pred)
fig, ax = plt.subplots(figsize=(10, 8))
ConfusionMatrixDisplay(cm, display_labels=clases).plot(
    ax=ax, cmap='Purples', colorbar=False, values_format='d',
    xticks_rotation=45)
ax.set_title(f'Matriz de Confusion — {NUM_CLASES} Clases Sanrio {DATASET_VERSION}',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(RUTA_MODELOS, f'confusion_matrix_{DATASET_VERSION}.png'), dpi=120)
plt.show()

print(f'\n{"="*58}')
print(f'  ACCURACY POR CLASE — Dataset {DATASET_VERSION}')
print(f'{"="*58}')
total_ok = 0
total_test_imgs = 0
resultados = []
for i, clase in enumerate(clases):
    mask = [r == i for r in y_real]
    preds = [p for p, m in zip(y_pred, mask) if m]
    ok = sum(1 for p in preds if p == i)
    tot = len(preds)
    acc = 100.0 * ok / tot if tot > 0 else 0
    total_ok += ok
    total_test_imgs += tot
    resultados.append((clase, ok, tot, acc))

for clase, ok, tot, acc in sorted(resultados, key=lambda x: x[3], reverse=True):
    barra = chr(9608) * max(1, int(acc/5))
    warning = '  REVISAR' if acc < 60 else ''
    print(f'  {clase:20s}: {ok:3d}/{tot:<3d} = {acc:5.1f}%  {barra}{warning}')

acc_global = 100.0 * total_ok / total_test_imgs
acc_promedio = sum(r[3] for r in resultados) / NUM_CLASES
print(f'{"-"*58}')
print(f'  Accuracy global:   {acc_global:.1f}%')
print(f'  Accuracy promedio: {acc_promedio:.1f}%')

print(f'\n  Pares mas confundidos:')
pares = [(cm[i][j], clases[i], clases[j])
         for i in range(NUM_CLASES) for j in range(NUM_CLASES) if i != j and cm[i][j] > 0]
for n, r, p in sorted(pares, reverse=True)[:6]:
    print(f'    {r:20s} → {p:20s}: {n}x')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELDA 10 — EXPORT ONNX FP32 + INT8                          ║
# ╚══════════════════════════════════════════════════════════════╝

try:
    import onnx
    from torch.ao.quantization import quantize_dynamic
    import shutil

    ruta_fp32 = os.path.join(RUTA_MODELS, f'tl_sanrio_{DATASET_VERSION}.onnx')
    ruta_int8 = os.path.join(RUTA_MODELS, f'tl_sanrio_{DATASET_VERSION}_int8.onnx')

    model.eval()
    dummy = torch.randn(1, 3, 224, 224).to(device)
    torch.onnx.export(model, dummy, ruta_fp32,
        input_names=['input'], output_names=['output'],
        dynamic_axes={'input': {0: 'batch'}, 'output': {0: 'batch'}},
        opset_version=17, export_params=True)
    mb_fp32 = os.path.getsize(ruta_fp32) / 1e6
    print(f'ONNX FP32: {mb_fp32:.1f} MB → {ruta_fp32}')

    model_int8 = quantize_dynamic(model.cpu(), {nn.Linear}, dtype=torch.qint8)
    torch.onnx.export(model_int8, torch.randn(1, 3, 224, 224), ruta_int8,
        input_names=['input'], output_names=['output'],
        dynamic_axes={'input': {0: 'batch'}, 'output': {0: 'batch'}},
        opset_version=17)
    mb_int8 = os.path.getsize(ruta_int8) / 1e6
    print(f'ONNX INT8: {mb_int8:.1f} MB → {ruta_int8}')
    print(f'Reduccion: {mb_fp32/mb_int8:.1f}x vs FP32')

    src = os.path.join(RUTA_MODELOS, f'clases_sanrio_{DATASET_VERSION}.json')
    dst = os.path.join(RUTA_MODELS, f'clases_sanrio_{DATASET_VERSION}.json')
    shutil.copy(src, dst)
    print(f'clases_sanrio_{DATASET_VERSION}.json copiado a models/')

except ModuleNotFoundError:
    print("onnx no instalado. pip install onnx onnxruntime")
    print(f'El .pth esta en: {ruta_pth}')
except Exception as e:
    print(f'Error ONNX: {e}')
    print(f'El .pth esta en: {ruta_pth}')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELDA 11 — EXPERIMENTO LOG                                  ║
# ║  Guarda en experimentos_tl.json para mantener historial      ║
# ╚══════════════════════════════════════════════════════════════╝

ruta_exp = os.path.join(RUTA_MODELOS, 'experimentos_tl.json')
if os.path.exists(ruta_exp):
    with open(ruta_exp, 'r') as f:
        experimentos = json.load(f)
else:
    experimentos = []

nuevo_exp = {
    'version': VERSION,
    'dataset': DATASET_VERSION,
    'fecha': datetime.now().strftime('%Y-%m-%d %H:%M'),
    'arquitectura': 'ResNet18',
    'num_clases': NUM_CLASES,
    'clases': clases,
    'mejor_accuracy': round(mejor_acc, 2),
    'accuracy_final': round(historial['test_acc'][-1], 2),
    'config': CONFIG,
    'dataset_stats': {
        'train': 2673,
        'test': 672,
        'duplicados_eliminados': 4,
        'leakage_eliminado': 0,
        'imagenes_<200px': 30,
    },
}

# Evitar duplicados (si se ejecuta 2 veces, reemplazar la entrada v4)
experimentos = [e for e in experimentos if e.get('version') != VERSION]
experimentos.append(nuevo_exp)

with open(ruta_exp, 'w', encoding='utf-8') as f:
    json.dump(experimentos, f, ensure_ascii=False, indent=2)
print(f'Experimento registrado en experimentos_tl.json')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELDA 12 — RESUMEN FINAL                                    ║
# ╚══════════════════════════════════════════════════════════════╝

import platform
try:
    import psutil
except:
    psutil = None

fin = time.time()
hh, rem = divmod(int(fin - INICIO), 3600)
mm, ss = divmod(rem, 60)

print()
print('='*62)
print(f'  Hecho con carino — {AUTORA}')
print(f'  Diplomado RNA * Modulo 4 * UAEM * Julio 2026')
print()
if torch.cuda.is_available():
    print(f'  GPU: {torch.cuda.get_device_name(0)}')
if psutil:
    print(f'  RAM: {psutil.virtual_memory().total/1e9:.1f} GB')
print(f'  PyTorch: {torch.__version__}')
print()
print(f'  Version:          {VERSION}')
print(f'  Dataset:          {DATASET_VERSION} ({NUM_CLASES} clases)')
print(f'  Tiempo total:     {hh}h {mm:02d}m {ss:02d}s')
print(f'  Accuracy global:  {acc_global:.2f}%')
print(f'  Accuracy promedio: {acc_promedio:.2f}%')
print(f'  Mejor accuracy:   {mejor_acc:.2f}%')
print(f'  Baseline v3:      66.43%')
print(f'  Mejora vs v3:     {mejor_acc - 66.43:+.2f}%')
print(f'  Fecha:            {datetime.now().strftime("%Y-%m-%d %H:%M")}')
print()
print('  Para deploy en HuggingFace Spaces:')
print(f'     models/tl_sanrio_{DATASET_VERSION}_int8.onnx')
print(f'     models/clases_sanrio_{DATASET_VERSION}.json')
print()
print('  Actualiza en app/config.py:')
print(f'     MODELO_PATH = "models/tl_sanrio_{DATASET_VERSION}_int8.onnx"')
print(f'     CLASES_PATH = "models/clases_sanrio_{DATASET_VERSION}.json"')
print('='*62)